# TACO Contamination Comparison

Comparing data contamination levels across datasets using the DCQ (Data Contamination Quiz) method.
TACO is a novel held-out dataset, and we expect it to show minimal contamination compared to established benchmarks.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
import matplotlib.patheffects as pe

# Publication-quality settings
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 11,
    'legend.fontsize': 9,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.8,
})

In [ ]:
# Load all contamination reports
bcq_dir = Path("phase3_bcq/gpt-5.2-2025-12-11")

reports = []
for f in bcq_dir.glob("*_contamination_report.json"):
    with open(f) as fp:
        reports.append(json.load(fp))

df = pd.DataFrame(reports)
df['mid_contamination'] = (df['min_contamination'] + df['max_contamination']) / 2
df['error'] = (df['max_contamination'] - df['min_contamination']) / 2
df = df.sort_values('mid_contamination', ascending=True)
df

In [ ]:
# Publication-quality figure
fig, ax = plt.subplots(figsize=(7, 4.5))

# Sort by contamination (ascending for bottom-to-top reading)
df_plot = df.sort_values('mid_contamination', ascending=True).reset_index(drop=True)

# Color palette - muted professional colors
NOVEL_COLOR = '#1a9850'      # Rich green for TACO
EXISTING_COLOR = '#4575b4'   # Steel blue for others

colors = [NOVEL_COLOR if d == 'TACO' else EXISTING_COLOR for d in df_plot['dataset']]
alphas = [1.0 if d == 'TACO' else 0.75 for d in df_plot['dataset']]

y_pos = np.arange(len(df_plot))

# Create bars with subtle styling
bars = ax.barh(y_pos, df_plot['mid_contamination'], 
               height=0.7,
               color=colors, 
               alpha=0.85,
               edgecolor='white', 
               linewidth=1.2)

# Error bars separately for cleaner look
ax.errorbar(df_plot['mid_contamination'], y_pos, 
            xerr=df_plot['error'],
            fmt='none', 
            ecolor='#333333',
            elinewidth=1.5, 
            capsize=3, 
            capthick=1.5,
            zorder=5)

# Highlight TACO with a subtle glow effect
taco_idx = df_plot[df_plot['dataset'] == 'TACO'].index[0]
bars[taco_idx].set_edgecolor(NOVEL_COLOR)
bars[taco_idx].set_linewidth(2)
bars[taco_idx].set_alpha(1.0)

# Labels
ax.set_yticks(y_pos)
ax.set_yticklabels(df_plot['dataset'], fontweight='medium')
ax.set_xlabel('Contamination Rate (%)', fontweight='medium')

# Make TACO label bold and green
labels = ax.get_yticklabels()
for i, label in enumerate(labels):
    if df_plot.iloc[i]['dataset'] == 'TACO':
        label.set_fontweight('bold')
        label.set_color(NOVEL_COLOR)

# Value labels at end of bars
for i, (mid, err, dataset) in enumerate(zip(df_plot['mid_contamination'], df_plot['error'], df_plot['dataset'])):
    fontweight = 'bold' if dataset == 'TACO' else 'normal'
    color = NOVEL_COLOR if dataset == 'TACO' else '#333333'
    ax.text(mid + err + 1.5, i, f'{mid:.1f}%', 
            va='center', ha='left',
            fontsize=9, fontweight=fontweight, color=color)

# Clean up axes
ax.set_xlim(0, 88)
ax.set_ylim(-0.5, len(df_plot) - 0.5)
ax.tick_params(axis='y', length=0)
ax.grid(axis='x', alpha=0.3, linestyle='-', linewidth=0.5)

# Legend
legend_elements = [
    Patch(facecolor=NOVEL_COLOR, alpha=0.85, edgecolor='white', linewidth=1.2, label='Novel Dataset (TACO)'),
    Patch(facecolor=EXISTING_COLOR, alpha=0.75, edgecolor='white', linewidth=1.2, label='Established Benchmarks'),
]
ax.legend(handles=legend_elements, loc='lower right', frameon=True, 
          framealpha=0.95, edgecolor='#cccccc', fancybox=False)

plt.tight_layout()
plt.savefig('taco_contamination_comparison.png', dpi=300, bbox_inches='tight', facecolor='white', pad_inches=0.1)
plt.savefig('taco_contamination_comparison.pdf', bbox_inches='tight', facecolor='white', pad_inches=0.1)
plt.show()

print(f"\nTACO shows {df_plot[df_plot['dataset']=='TACO']['mid_contamination'].values[0]:.1f}% contamination")
print(f"vs. {df_plot[df_plot['dataset']!='TACO']['mid_contamination'].mean():.1f}% average for established datasets")

In [ ]:
# Create a cleaner version for paper
fig, ax = plt.subplots(figsize=(8, 5))

# Sort by contamination and create color scheme
df_sorted = df.sort_values('mid_contamination', ascending=False)

colors = ['#2ecc71' if d == 'TACO' else '#e74c3c' if m > 50 else '#f39c12' if m > 30 else '#3498db' 
          for d, m in zip(df_sorted['dataset'], df_sorted['mid_contamination'])]

y_pos = np.arange(len(df_sorted))
bars = ax.barh(y_pos, df_sorted['mid_contamination'], xerr=df_sorted['error'],
               color=colors, edgecolor='white', linewidth=0.8,
               capsize=3, error_kw={'elinewidth': 1.2, 'capthick': 1.2, 'color': '#2c3e50'})

ax.set_yticks(y_pos)
ax.set_yticklabels(df_sorted['dataset'], fontsize=11)
ax.set_xlabel('Contamination Rate (%)', fontsize=12)
ax.set_title('Data Contamination in GPT-5.2 (DCQ Method)', fontsize=13, fontweight='bold', pad=10)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', label='Novel (TACO)'),
    Patch(facecolor='#3498db', label='Low (<30%)'),
    Patch(facecolor='#f39c12', label='Medium (30-50%)'),
    Patch(facecolor='#e74c3c', label='High (>50%)')
]
ax.legend(handles=legend_elements, loc='lower right', framealpha=0.9, fontsize=9)

ax.set_xlim(0, 85)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('taco_contamination_comparison_paper.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('taco_contamination_comparison_paper.pdf', bbox_inches='tight', facecolor='white')
plt.show()